In [ ]:
!pip install groq chromadb sentence-transformers pandas -q

print("All packages installed successfully")

All packages installed successfully


In [ ]:
import pandas as pd
import sqlite3
from groq import Groq
import chromadb
import os

In [ ]:
GROQ_API_KEY="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"
client=Groq(api_key=GROQ_API_KEY)
MODEL="llama-3.1-8b-instant"

print("Groq client configured")
print(f"Model : {MODEL}")
print("Status:Ready to generate AI Response")

Groq client configured
Model : llama-3.1-8b-instant
Status:Ready to generate AI Response


In [ ]:
responsible_system_prompt="""
You are a helpful data analysis assistant.
You only answer questions based on the data provided to you.
If you are not sure about something, say: I do not have enough information to answer that accurately.
Never make up statistics or facta that are not in the data you receive.
"""

test_response=client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role":"system","content":responsible_system_prompt},
        {"role":"user","content":"What is the population on Earth?"}
    ]
)

ai_answer=test_response.choices[0].message.content

print("=== Responsible AI Text ===")
print("Question : What is the population on Earth?")
print("AI Response (with guardrail): ")
print("Answer : ",ai_answer)
print("\n a Responsible AI should admit it cannot answer accurately")

=== Responsible AI Text ===
Question : What is the population on Earth?
AI Response (with guardrail): 
Answer :  I don't have enough information to answer that accurately. The population of Earth can vary over time. If you can provide historical or current population data, I can give you a more accurate answer.

 a Responsible AI should admit it cannot answer accurately


In [ ]:
student_df=pd.read_csv("student_performance (3).csv")
print("=== Student Reference Dataset ===")
print(f"Shape: {student_df.shape[0]} students, {student_df.shape[1]} columns")
print()
print(student_df.head(5))

=== Student Reference Dataset ===
Shape: 30 students, 11 columns

   student_id          name  age  gender branch  attendance_pct  \
0           1  Aarav Sharma   20    Male    CSE              85   
1           2   Priya Patel   21  Female    ECE              92   
2           3   Rohit Kumar   20    Male   MECH              67   
3           4    Sneha Iyer   22  Female    CSE              95   
4           5  Vikram Singh   21    Male  CIVIL              72   

   assignment_score  midterm_score  final_score  gpa passed  
0                78             72           76  7.6    Yes  
1                88             85           89  8.9    Yes  
2                55             60           58  5.8    Yes  
3                92             90           94  9.4    Yes  
4                62             65           63  6.3    Yes  


In [ ]:
notes_df=pd.read_csv("college_notes (1).csv")
print("=== College Notes Dataset ===")
print(f"Shape: {notes_df.shape[0]} notes, {notes_df.shape[1]} columns")
print()
print(notes_df[['note_id','subject','topic','difficulty']].to_string(index=False))


=== College Notes Dataset ===
Shape: 15 notes, 6 columns

 note_id             subject                       topic   difficulty
       1     Data Structures                      Arrays     Beginner
       2     Data Structures                Linked Lists     Beginner
       3     Data Structures                Binary Trees Intermediate
       4     Data Structures           Stacks and Queues     Beginner
       5 Database Management                  SQL Basics     Beginner
       6 Database Management               Normalization Intermediate
       7 Database Management                    Indexing Intermediate
       8    Machine Learning                  Regression Intermediate
       9    Machine Learning              Classification Intermediate
      10    Machine Learning                  Clustering     Advanced
      11  Python Programming                   Functions     Beginner
      12  Python Programming Object Oriented Programming Intermediate
      13  Python Programming    

In [ ]:
print("=== Data quality Report:student_performance (3).csv ===")
print("Missing values per column:")
print(student_df.isnull().sum())
print()

print("Data Types:")
print(student_df.dtypes)
print()

print(f"Duplicate rows: {student_df.duplicated().sum()}")
print("Data quality check complete.")

=== Data quality Report:student_performance (3).csv ===
Missing values per column:
student_id          0
name                0
age                 0
gender              0
branch              0
attendance_pct      0
assignment_score    0
midterm_score       0
final_score         0
gpa                 0
passed              0
dtype: int64

Data Types:
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object

Duplicate rows: 0
Data quality check complete.


In [ ]:
conn=sqlite3.connect(':memory:')
student_df.to_sql('students',conn,if_exists='replace',index=False)

print("SQL Database created")
print("Table 'students' loaded with",len(student_df),"rows")


SQL Database created
Table 'students' loaded with 30 rows


In [ ]:
query1="""
SELECT
  branch,
  COUNT(*) AS total_students,
  ROUND(AVG(gpa),2) AS avg_gpa,
  ROUND(AVG(attendance_pct),1) AS avg_attendance
FROM students
GROUP BY branch
ORDER BY avg_gpa DESC
"""

branch_analysis=pd.read_sql(query1,conn)
print("=== Branch-wise Performance Analysis ===")
print(branch_analysis.to_string(index=False))

=== Branch-wise Performance Analysis ===
branch  total_students  avg_gpa  avg_attendance
    IT               5     8.64            89.4
   CSE              10     7.42            80.0
  MECH               5     7.22            79.4
 CIVIL               4     6.75            75.0
   ECE               6     6.38            69.2


In [ ]:
query2="""
SELECT name,branch,gpa,attendance_pct,passed
FROM students
ORDER BY gpa DESC
LIMIT 5
"""

top_students=pd.read_sql(query2,conn)
print("=== Top 5 Students ===")
print(top_students.to_string(index=False))


=== Top 5 Students ===
            name branch  gpa  attendance_pct passed
  Meera Krishnan     IT  9.5              96    Yes
      Sneha Iyer    CSE  9.4              95    Yes
Lakshmi Chandran    CSE  9.2              94    Yes
    Swathi Menon     IT  9.1              93    Yes
     Priya Patel    ECE  8.9              92    Yes


In [ ]:
query3="""
SELECT
  passed,
  COUNT(*) AS student_count,
  ROUND(AVG(gpa),2) AS avg_gpa
FROM students
GROUP BY passed
"""

pass_fail=pd.read_sql(query3,conn)
print("=== Pass/Fail Analysis ===")
print(pass_fail.to_string(index=False))


total=len(student_df)
passed=len(student_df[student_df['passed']=='Yes'])
pass_rate=round((passed/total)*100,1)


print(f"Overall Pass rate:{pass_rate}% ({passed}/{total} students)")

=== Pass/Fail Analysis ===
passed  student_count  avg_gpa
    No              3     4.47
   Yes             27     7.61
Overall Pass rate:90.0% (27/30 students)


In [ ]:
branch_summary=""
for _,row in branch_analysis.iterrows():
  branch_summary+=f" -{row['branch']}:{row['total_students']} students, Avg GPA {row['avg_gpa']},Avg Attendance {row['avg_attendance']}\n"

top_summary=""
for _, row in top_students.iterrows():
  top_summary+=f" ={row['name']} ({row['branch']}): GPA {row['gpa']}\n"

data_summary=f"""
STUDENT PERFOEMANCE DATA SUMMARY
Total students:{total}
Overall Pass Rate:{pass_rate}%
Average GPA across all students:{round(student_df['gpa'].mean(),2)}

Performance by Branch:
{branch_summary}

Top 5 Students:
{top_summary}
"""

print("=== Student Performance Data Summary ===")
print(data_summary)

=== Student Performance Data Summary ===

STUDENT PERFOEMANCE DATA SUMMARY
Total students:30
Overall Pass Rate:90.0%
Average GPA across all students:7.29

Performance by Branch:
 -IT:5 students, Avg GPA 8.64,Avg Attendance 89.4
 -CSE:10 students, Avg GPA 7.42,Avg Attendance 80.0
 -MECH:5 students, Avg GPA 7.22,Avg Attendance 79.4
 -CIVIL:4 students, Avg GPA 6.75,Avg Attendance 75.0
 -ECE:6 students, Avg GPA 6.38,Avg Attendance 69.2


Top 5 Students:
 =Meera Krishnan (IT): GPA 9.5
 =Sneha Iyer (CSE): GPA 9.4
 =Lakshmi Chandran (CSE): GPA 9.2
 =Swathi Menon (IT): GPA 9.1
 =Priya Patel (ECE): GPA 8.9




In [ ]:
system_prompt="""
You are an expert academic data analyist working for an engineering college.
You receive student performance summaries and provide clear, actionable insights.
Always base your analysis strictly on the data provided.
If you are uncertain about something, say so clearly.
Format your response with numbered ppoints for clarity
"""

user_message=f"""
Here is the student performance summary:
{data_summary}
"""

response=client.chat.completions.create(
    model=MODEL,
    max_tokens=600,
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_message}
    ]
)

ai_analysis=response.choices[0].message.content

print("="*60)
print("AI-Generated Data Analysis")
print("="*60)
print(ai_analysis)


AI-Generated Data Analysis
**Student Performance Analysis Report**

Based on the provided data, here are the insights and observations:

1. **Overall Performance Trends**: The overall pass rate is 90.0%, indicating a high pass rate among students. The average GPA across all students is 7.29, which suggests that students in general are performing well academically.

2. **Performance by Branch**: The data suggests that there are significant variations in student performance across branches.
   - **IT students** appear to be performing well, with an average GPA of 8.64 and a high attendance rate of 89.4%.
   - **CSE students** have a moderate average GPA of 7.42 and an average attendance rate of 80.0%.
   - **MECH and CIVIL branches** show relatively lower average GPAs (7.22 and 6.75 respectively) and lower attendance rates (79.4% and 75.0% respectively).
   - **ECE students** have the lowest average GPA (6.38) and attendance rate (69.2%).

3. **Branch-wise Performance Comparison**:
   - 